# Module 3 - Data Project

In [1]:
#install packages
install.packages("tableone")
install.packages("MatchIt")
library(tableone)
library(MatchIt)

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [2]:
# two versions of lalonde data, get the right one
data(lalonde)
View(lalonde)

,treat,age,educ,race,married,nodegree,re74,re75,re78
,<int>,<int>,<int>,<fct>,<int>,<int>,<dbl>,<dbl>,<dbl>
NSW1,1,37,11,black,1,1,0,0,9930.0460
NSW2,1,22,9,hispan,0,1,0,0,3595.8940
NSW3,1,30,12,black,0,0,0,0,24909.4500
NSW4,1,27,11,black,0,1,0,0,7506.1460
NSW5,1,33,8,black,0,1,0,0,289.7899
NSW6,1,22,9,black,0,1,0,0,4056.4940
NSW7,1,23,12,black,0,0,0,0,0.0000
NSW8,1,32,11,black,0,1,0,0,8472.1580
NSW9,1,22,16,black,0,0,0,0,2164.0220


In [3]:
xvars<-c("age","educ","race","married","nodegree","re74",
         "re75")
matchedtab1<-CreateTableOne(vars=xvars, strata ="treat", 
                            data=lalonde, test = FALSE)
print(matchedtab1, smd = TRUE)

                      Stratified by treat
                       0                 1                 SMD   
  n                        429               185                 
  age (mean (SD))        28.03 (10.79)     25.82 (7.16)     0.242
  educ (mean (SD))       10.24 (2.86)      10.35 (2.01)     0.045
  race (%)                                                  1.701
     black                  87 (20.3)        156 (84.3)          
     hispan                 61 (14.2)         11 ( 5.9)          
     white                 281 (65.5)         18 ( 9.7)          
  married (mean (SD))     0.51 (0.50)       0.19 (0.39)     0.719
  nodegree (mean (SD))    0.60 (0.49)       0.71 (0.46)     0.235
  re74 (mean (SD))     5619.24 (6788.75) 2095.57 (4886.62)  0.596
  re75 (mean (SD))     2466.48 (3292.00) 1532.06 (3219.25)  0.287


In [7]:
#unbalaced outcome analysis
y_trt<-lalonde$re78[lalonde$treat==1]
y_con<-lalonde$re78[lalonde$treat==0]

#pairwise difference
diffy<-mean(y_trt)-mean(y_con)
print(diffy)

[1] -635.0262


In [10]:
# Fit a model on the confounders only, not the outcome value re78
ps1model<-glm(treat ~ age+educ+race+married+nodegree+re74+re75,
    family=binomial, data=lalonde)

#show coefficients etc
summary(ps1model)
#create propensity score
ps1core<-ps1model$fitted.values
print(min(ps1core))
print(max(ps1core))


Call:
glm(formula = treat ~ age + educ + race + married + nodegree + 
    re74 + re75, family = binomial, data = lalonde)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)    
(Intercept) -1.663e+00  9.709e-01  -1.713  0.08668 .  
age          1.578e-02  1.358e-02   1.162  0.24521    
educ         1.613e-01  6.513e-02   2.477  0.01325 *  
racehispan  -2.082e+00  3.672e-01  -5.669 1.44e-08 ***
racewhite   -3.065e+00  2.865e-01 -10.699  < 2e-16 ***
married     -8.321e-01  2.903e-01  -2.866  0.00415 ** 
nodegree     7.073e-01  3.377e-01   2.095  0.03620 *  
re74        -7.178e-05  2.875e-05  -2.497  0.01253 *  
re75         5.345e-05  4.635e-05   1.153  0.24884    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 751.49  on 613  degrees of freedom
Residual deviance: 487.84  on 605  degrees of freedom
AIC: 505.84

Number of Fisher Scoring iterations: 5


[1] 0.009080193
[1] 0.8531528


In [19]:
# write.csv(lalonde, "~/work/lalonde-matchit.csv", row.names=FALSE)

## Follow up to quiz data problems

In [12]:
# create a column oriented standard difference funciton
std_diff <- function(dfr, tcol, gcol='treat'){
    xt <- dfr[dfr[[gcol]]==1,tcol]
    xc <- dfr[dfr[[gcol]]==0,tcol]
    nt = 2; nc = 2;  
    #nt = length(xt);nc = length(xc);
    (mean(xt) - mean(xc))/(sqrt( ((nt-1)*var(xt) + (nc-1)*var(xc))/(nt+nc-2)))
}

In [14]:
install.packages("Matching")
library(Matching)

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Loading required package: MASS

## 
##  Matching (Version 4.10-15, Build Date: 2024-10-14)
##  See https://www.jsekhon.com for additional documentation.
##  Please cite software as:
##   Jasjeet S. Sekhon. 2011. ``Multivariate and Propensity Score Matching
##   Software with Automated Balance Optimization: The Matching package for R.''
##   Journal of Statistical Software, 42(7): 1-52. 
##




In [16]:
fit <- glm(treat ~ age + educ + black + hispan + married + nodegree + re74 + re75,
           data=lalonde, family = binomial(link = "logit"))
prop_hat <- predict(fit, newdata = lalonde, type="response")

ERROR: Error in eval(predvars, data, env): object 'black' not found


In [15]:
set.seed(931139)
pmatch <- Match(Tr = lalonde$treat, M=1, X=lalonde$pscore, replace=FALSE, caliper=NaN)
matched <- lalonde[unlist(pmatch[c('index.treated', 'index.control')]), ]

ERROR: Error in array(x, c(length(x), 1L), if (!is.null(names(x))) list(names(x), : 'data' must be of a vector type, was 'NULL'
